# 🏛️ Prediksi Ketertagihan WP — V2 (Kapasitas & Kemauan Bayar)

Notebook ini adalah **baseline prediktif capacity-only** untuk model ketertagihan.

**Perbedaan fundamental terhadap V1:**
- Input V2 berisi **seluruh kohir** milik 2.873 WP (kohir berlabel + belum berlabel).
- `LABEL` tetap 3 kelas: **0 = Rendah, 1 = Sedang, 2 = Tinggi**.
- Kohir tanpa label tetap dipakai untuk membentuk profil/portofolio WP, tetapi tidak ikut menentukan target.
- Fitur pembentuk label (`SETOR_*`, pencairan/sisa, respons/tindakan penagihan) **dilarang masuk model**.
- Model hanya memakai sinyal **kapasitas & kemauan bayar yang independen dari outcome**: profil tunggakan, status WP, SPT/omzet, faktur, sektor, dan kronologi dasar utang.

**Tujuan bisnis:** memprediksi WP yang tunggakannya realistis tertagih — bukan sekadar menghitung ulang rumus label.

> ⚠️ **Batas validitas:** LABEL adalah outcome administratif historis pada snapshot, belum target masa-depan dengan cutoff waktu yang ketat. Tanpa tanggal pembayaran/cutoff historis, V2 adalah baseline cross-sectional yang lebih jujur daripada V1, tetapi belum bukti validasi prospektif. Data aset (rekening, kendaraan, properti, piutang/aset teridentifikasi) juga belum tersedia; aktivitas ekonomi dipakai sebagai proksi kapasitas.

## ⚙️ Setup

In [ ]:
import json
import os

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay,
    f1_score, balanced_accuracy_score, recall_score,
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RNG = 42
np.random.seed(RNG)
pd.set_option("display.max_columns", 80)

CONFIG = {
    "path_utama": "../dataset/SAMPLE_DATA_ENRICH_V2.csv",
    "snapshot_training": "2026-08-20",
    "path_metrics": "../dataset/taxpayer_metrics_result.csv",
    "path_metrics_fallback": "../dataset/processed/wp_features.csv",
    "path_customer": "../dataset/data_customer.json",
    "path_supplier": "../dataset/data_supplier.json",
    "path_features": "../dataset/processed/wp_features_v2.csv",
    "path_model": "../models/ketertagihan_wp_v2.joblib",
    "path_predictions": "../dataset/processed/test_predictions_v2.csv",
    "label_names": {0: "Rendah", 1: "Sedang", 2: "Tinggi"},
    "test_size": 0.20,
    "cv_folds": 5,
}

print("Setup selesai ✅")

## 1. 📥 Memuat Data

Metrik SPT/omzet adalah fitur inti kapasitas bayar. Notebook memakai
`taxpayer_metrics_result.csv`; bila file itu tidak tersedia, fallback dibaca dari
hasil V1 (`processed/wp_features.csv`) untuk 2.873 WP yang sama.

In [ ]:
df = pd.read_csv(CONFIG["path_utama"], sep=";", dtype=str)
df.columns = df.columns.str.strip()

if os.path.exists(CONFIG["path_metrics"]):
    met = pd.read_csv(CONFIG["path_metrics"], dtype=str)
    met_key = "NPWP"
    print("Metrik SPT: sumber utama")
elif os.path.exists(CONFIG["path_metrics_fallback"]):
    met = pd.read_csv(CONFIG["path_metrics_fallback"], dtype=str,
                      usecols=["NPWP16", "RASIO_LAPOR_SPT_3THN",
                               "FLAG_LAPOR_SPT_TERAKHIR", "PEREDARAN_BRUTO"])
    met_key = "NPWP16"
    print("⚠️ Metrik SPT: fallback dari processed/wp_features.csv V1")
else:
    raise FileNotFoundError("Metrik SPT/omzet tidak ditemukan; fitur kapasitas utama hilang.")

cust = pd.DataFrame(json.load(open(CONFIG["path_customer"], encoding="utf-8")))
supp = pd.DataFrame(json.load(open(CONFIG["path_supplier"], encoding="utf-8")))

print(f"V2 mentah : {df.shape[0]:,} kohir × {df.shape[1]} kolom")
print(f"WP unik   : {df['NPWP16'].str.strip().nunique():,}")
print("LABEL     :", df["LABEL"].value_counts(dropna=False).to_dict())

## 2. 🧹 Pembersihan & Normalisasi

- NPWP disimpan sebagai string 16 digit.
- Duplikat penuh dibuang.
- Tanggal dan numerik dikonversi eksplisit.
- Kode KPP/KLU dirapikan.
- Fitur kronologi dasar dihitung sebelum agregasi.

Tidak ada filter populasi tambahan — populasi ditentukan oleh tarikan V2.

In [ ]:
def norm_npwp(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.zfill(16)

n_raw = len(df)
df["NPWP16"] = norm_npwp(df["NPWP16"])
df = df.drop_duplicates().copy()

for c in df.select_dtypes(include=["object", "str"]).columns:
    df[c] = df[c].str.strip()

DATE_COLS = [
    "TGL_PRODUK_HUKUM", "TGL_UTANG_DPT_DITAGIH", "TGL_INKRAH", "TGL_DALUWARSA",
    "TGL_TEGURAN", "TGL_PENYAMPAIAN_SP", "TGL_BAPS", "TGL_KMK_CEGAH", "TGL_SPRINDRA",
]
for c in DATE_COLS:
    df[c] = pd.to_datetime(df[c], errors="coerce")

NUM_COLS = [
    "LABEL", "FG_INKRAH_CTX", "TH_PJK", "THN_DALUARSA",
    "NILAI_STPSKP", "NILAI_SISA", "SETOR_SEBELUM_COLL_DATE", "SETOR_SEBELUM_TEGURAN",
    "SETOR_TEGURAN", "SETOR_PAKSA", "SETOR_SITA", "SETOR_CEGAH", "SETOR_SPRINDRA",
    "JML_SURAT_TEGURAN", "JML_SURAT_PAKSA", "FLAG_PERNAH_DISITA",
    "FLAG_PERNAH_BLOKIR", "FLAG_RESPON_PENAGIHAN", "NILAI_CAIR_HISTORIS",
]
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

assert df["LABEL"].dropna().isin([0, 1, 2]).all()

df["JENIS_KPP_BKM"] = df["JENIS_KPP_BKM"].replace(
    {"P": "PRATAMA", "M": "MADYA", "B": "BESAR", "K": "KHUSUS"}
)
df["SEKTOR_KLU"] = df["KD_KLU"].str.zfill(5).str[0].map({
    "0": "PERTANIAN", "1": "PERTAMBANGAN", "2": "INDUSTRI", "3": "ENERGI/AIR",
    "4": "KONSTRUKSI", "5": "PERDAGANGAN", "6": "TRANSPORTASI", "7": "INFORMASI",
    "8": "JASA", "9": "JASA_LAIN",
}).fillna("LAINNYA")
df["KODE_KETETAPAN"] = df["NO_STPSKP"].str.split("/").str[1].fillna("UNKNOWN")
df["SELISIH_TAHUN_TERBIT"] = df["TGL_PRODUK_HUKUM"].dt.year - df["TH_PJK"]

# Kontrak artefak V2: snapshot training dibekukan agar rerun reproducible.
# Nama fitur historis tetap UMUR_UTANG_*, tetapi sumbernya TGL_PRODUK_HUKUM.
SNAPSHOT = pd.Timestamp(CONFIG["snapshot_training"])
df["UMUR_UTANG_HARI"] = (SNAPSHOT - df["TGL_PRODUK_HUKUM"]).dt.days
df["SISA_DALUWARSA_HARI"] = (df["TGL_DALUWARSA"] - SNAPSHOT).dt.days

print(f"Baris: {n_raw:,} → {len(df):,} (duplikat dibuang: {n_raw-len(df):,})")
print(f"Snapshot: {SNAPSHOT.date()}")
print("LABEL setelah dedup:", df["LABEL"].value_counts(dropna=False).to_dict())

## 3. ✅ Validasi Rumus LABEL

LABEL V2 harus identik dengan rumus sumber:

- `SETOR_TEGURAN > 0` → 2 (Tinggi)
- jika tidak, `SETOR_PAKSA > 0 OR SETOR_SITA > 0` → 1 (Sedang)
- selain itu → 0 (Rendah)
- `TGL_TEGURAN IS NULL` → LABEL NULL (kohir konteks, belum punya outcome)

Kolom pembentuk rumus hanya dipakai untuk **validasi ini**, tidak masuk model.

In [ ]:
berlabel = df["LABEL"].notna()
label_rule = np.select(
    [df["SETOR_TEGURAN"].fillna(0) > 0,
     (df["SETOR_PAKSA"].fillna(0) > 0) | (df["SETOR_SITA"].fillna(0) > 0)],
    [2, 1], default=0,
)

agreement = (label_rule[berlabel] == df.loc[berlabel, "LABEL"]).mean()
assert agreement == 1.0, "LABEL V2 tidak 100% sesuai rumus sumber!"
assert (df["LABEL"].isna() == df["TGL_TEGURAN"].isna()).all(),        "Pola LABEL NULL tidak identik dengan TGL_TEGURAN NULL."

print(f"Kohir berlabel    : {berlabel.sum():,}")
print(f"Kohir tanpa label : {(~berlabel).sum():,} (konteks fitur saja)")
print(f"Kesesuaian rumus  : {agreement*100:.1f}% ✅")

## 4. 🏷️ Target WP dari Kohir Berlabel

Target utama V2 tetap mengikuti keputusan V1:
- `LABEL` WP = **modus berdasarkan jumlah kohir berlabel**;
- jika seri, ambil label terendah/terburuk (konservatif).

Notebook juga menghitung target alternatif **tertimbang nilai ketetapan** untuk diagnostik.
Perbedaan keduanya penting: bila satu kohir sangat besar tetapi jumlahnya kalah, mode hitungan
bisa tidak merepresentasikan risiko rupiah. Alternatif ini disimpan, tetapi **tidak menjadi target model** sebelum ada keputusan domain.

`LABEL_PURITY` = proporsi kohir yang masuk kelas mayoritas. Nilai rendah menandakan target WP heterogen/noisy.

In [ ]:
lab = (
    df[df["LABEL"].notna()]
      .groupby(["NPWP16", "LABEL"], as_index=False)
      .agg(JML=("LABEL", "size"), NILAI=("NILAI_STPSKP", "sum"))
)

# Modus hitungan; seri -> LABEL terkecil (terburuk)
label_mode = (
    lab.sort_values(["NPWP16", "JML", "LABEL"], ascending=[True, False, True])
       .drop_duplicates("NPWP16")
       .set_index("NPWP16")["LABEL"]
       .rename("LABEL")
)
# Alternatif: kelas dengan total nilai ketetapan terbesar; seri -> terburuk
label_value = (
    lab.sort_values(["NPWP16", "NILAI", "LABEL"], ascending=[True, False, True])
       .drop_duplicates("NPWP16")
       .set_index("NPWP16")["LABEL"]
       .rename("LABEL_VALUE_WEIGHTED")
)

label_meta = lab.groupby("NPWP16").agg(
    JML_KOHIR_BERLABEL=("JML", "sum"),
    MAX_KOHIR_SEKELAS=("JML", "max"),
).join(label_mode).join(label_value)
label_meta["LABEL_PURITY"] = label_meta["MAX_KOHIR_SEKELAS"] / label_meta["JML_KOHIR_BERLABEL"]

assert len(label_meta) == df["NPWP16"].nunique(), "Ada WP tanpa kohir berlabel."

print("Distribusi target modus       :", label_meta["LABEL"].value_counts().sort_index().to_dict())
print("Distribusi target tertimbang  :", label_meta["LABEL_VALUE_WEIGHTED"].value_counts().sort_index().to_dict())
print(f"Agreement kedua definisi      : {(label_meta['LABEL']==label_meta['LABEL_VALUE_WEIGHTED']).mean()*100:.1f}%")
display(label_meta["LABEL_PURITY"].describe(percentiles=[.25, .5, .75, .9]).round(3).to_frame())

fig, ax = plt.subplots(figsize=(5, 3))
label_meta["LABEL_PURITY"].hist(bins=20, ax=ax, color="slateblue")
ax.set_title("Kemurnian label WP (modus kohir)")
ax.set_xlabel("Proporsi kohir di kelas mayoritas"); ax.set_ylabel("Jumlah WP")
plt.tight_layout(); plt.show()

## 5. 📦 Agregasi Fitur Kapasitas/Profil per WP

Semua kohir (termasuk LABEL NULL) dipakai untuk menggambarkan portofolio WP.

Fitur agregat yang digunakan:
- profil WP & sektor;
- total/rata-rata/median/maksimum tunggakan;
- jumlah/ragam kohir dan pajak;
- umur sejak produk hukum (nama historis fitur: `UMUR_UTANG_*`) & sisa daluwarsa;
- jenis pajak/ketetapan dominan.

**Tidak diagregasi menjadi fitur:** pembayaran, pencairan/sisa, respons, surat paksa,
sita/blokir, dan tanggal tindakan — semuanya berkaitan langsung/tidak langsung dengan outcome LABEL.

In [ ]:
def mode_aman(s):
    m = s.mode(dropna=True)
    return m.iloc[0] if not m.empty else np.nan

wp = (
    df.groupby("NPWP16")
      .agg(
          # Mode menghindari ketergantungan urutan baris bila atribut WP tidak konsisten antar kohir.
          NAMA_WP=("NAMA_WP", mode_aman),
          STS_WP=("STS_WP", mode_aman),
          JENIS_WP=("JENIS_WP", mode_aman),
          JENIS_KPP_BKM=("JENIS_KPP_BKM", mode_aman),
          KD_KANWIL=("KD_KANWIL", mode_aman),
          KD_KLU=("KD_KLU", mode_aman),
          SEKTOR_KLU=("SEKTOR_KLU", mode_aman),
          JENIS_PAJAK_DOMINAN=("JENIS_PAJAK", mode_aman),
          KODE_KETETAPAN_DOMINAN=("KODE_KETETAPAN", mode_aman),
          TOTAL_TUNGGAKAN_POKOK=("NILAI_STPSKP", "sum"),
          RATA_TUNGGAKAN=("NILAI_STPSKP", "mean"),
          MEDIAN_TUNGGAKAN=("NILAI_STPSKP", "median"),
          MAX_TUNGGAKAN=("NILAI_STPSKP", "max"),
          JML_KETETAPAN=("NO_STPSKP", "count"),
          JML_JENIS_PAJAK=("JENIS_PAJAK", "nunique"),
          JML_JENIS_KETETAPAN=("KODE_KETETAPAN", "nunique"),
          UMUR_UTANG_RATA=("UMUR_UTANG_HARI", "mean"),
          UMUR_UTANG_TERTUA=("UMUR_UTANG_HARI", "max"),
          SISA_DALUWARSA_TERDEKAT=("SISA_DALUWARSA_HARI", "min"),
          RATA_SELISIH_TAHUN_TERBIT=("SELISIH_TAHUN_TERBIT", "mean"),
      )
      .join(label_meta[["LABEL", "LABEL_VALUE_WEIGHTED", "LABEL_PURITY", "JML_KOHIR_BERLABEL"]])
      .reset_index()
)

jml_total = df.groupby("NPWP16").size()
wp["JML_KOHIR_TANPA_LABEL"] = wp["NPWP16"].map(jml_total) - wp["JML_KOHIR_BERLABEL"]

print(f"WP teragregasi: {wp.shape[0]:,} baris × {wp.shape[1]} kolom")
wp[["NPWP16", "JML_KETETAPAN", "JML_KOHIR_BERLABEL", "JML_KOHIR_TANPA_LABEL",
    "TOTAL_TUNGGAKAN_POKOK", "LABEL", "LABEL_PURITY"]].head()

## 6. 🔗 Gabung Data SPT & Faktur

- SPT/omzet: kapasitas dan kepatuhan.
- Faktur customer/supplier: aktivitas ekonomi & jejaring transaksi.
- Missing SPT dibuat sebagai flag eksplisit (`FLAG_TANPA_DATA_SPT`) sebelum imputasi.

In [ ]:
# Normalisasi metrik SPT (mendukung sumber utama dan fallback V1)
met["NPWP16"] = norm_npwp(met[met_key])
met = met.rename(columns={
    "rasio_lapor_spt_3thn": "RASIO_LAPOR_SPT_3THN",
    "flag_lapor_spt_terakhir": "FLAG_LAPOR_SPT_TERAKHIR",
    "peredaran_bruto": "PEREDARAN_BRUTO",
})
for c in ["RASIO_LAPOR_SPT_3THN", "FLAG_LAPOR_SPT_TERAKHIR", "PEREDARAN_BRUTO"]:
    met[c] = pd.to_numeric(met[c], errors="coerce")

cust["NPWP16"] = norm_npwp(cust["npwp"])
supp["NPWP16"] = norm_npwp(supp["npwp"])
cust = cust.rename(columns={"jml_customer": "JML_CUSTOMER", "total_jml_dpp": "DPP_CUSTOMER",
                            "total_jml_faktur": "FAKTUR_CUSTOMER"})
supp = supp.rename(columns={"jml_supplier": "JML_SUPPLIER", "total_jml_dpp": "DPP_SUPPLIER",
                            "total_jml_faktur": "FAKTUR_SUPPLIER"})

wp = (
    wp.merge(met[["NPWP16", "RASIO_LAPOR_SPT_3THN", "FLAG_LAPOR_SPT_TERAKHIR", "PEREDARAN_BRUTO"]],
             on="NPWP16", how="left", validate="one_to_one")
      .merge(cust[["NPWP16", "JML_CUSTOMER", "DPP_CUSTOMER", "FAKTUR_CUSTOMER"]],
             on="NPWP16", how="left", validate="one_to_one")
      .merge(supp[["NPWP16", "JML_SUPPLIER", "DPP_SUPPLIER", "FAKTUR_SUPPLIER"]],
             on="NPWP16", how="left", validate="one_to_one")
)

wp["FLAG_TANPA_DATA_SPT"] = wp["RASIO_LAPOR_SPT_3THN"].isna().astype(int)
wp["FLAG_PEREDARAN_NOL_KOSONG"] = (wp["PEREDARAN_BRUTO"].isna() | (wp["PEREDARAN_BRUTO"] <= 0)).astype(int)
wp["RASIO_TUNGGAKAN_PEREDARAN"] = np.where(
    wp["PEREDARAN_BRUTO"] > 0,
    wp["TOTAL_TUNGGAKAN_POKOK"] / wp["PEREDARAN_BRUTO"],
    np.nan,
)

for src, dst in [
    ("TOTAL_TUNGGAKAN_POKOK", "LOG_TOTAL_TUNGGAKAN"),
    ("MAX_TUNGGAKAN", "LOG_MAX_TUNGGAKAN"),
    ("PEREDARAN_BRUTO", "LOG_PEREDARAN_BRUTO"),
    ("DPP_CUSTOMER", "LOG_DPP_CUSTOMER"),
    ("DPP_SUPPLIER", "LOG_DPP_SUPPLIER"),
]:
    wp[dst] = np.log1p(wp[src].clip(lower=0))
wp["TOTAL_MITRA"] = wp["JML_CUSTOMER"] + wp["JML_SUPPLIER"]

print("Missing setelah merge:", wp[["RASIO_LAPOR_SPT_3THN", "JML_CUSTOMER", "JML_SUPPLIER"]].isna().sum().to_dict())
print("Dimensi:", wp.shape)

## 7. 🛡️ Kontrak Anti-Leakage

Daftar `OUTCOME_COLS` adalah kolom yang membentuk atau sangat dekat dengan LABEL.
Kolom tersebut boleh ada di data mentah untuk validasi, tetapi **tidak boleh** ada di `FEATURES`.

Model V2 sengaja lebih sulit daripada V1: performa yang lebih rendah adalah estimasi
prediktif yang lebih jujur, bukan kemunduran.

In [ ]:
OUTCOME_COLS = [
    "NILAI_SISA", "NILAI_CAIR_HISTORIS",
    "SETOR_SEBELUM_COLL_DATE", "SETOR_SEBELUM_TEGURAN", "SETOR_TEGURAN",
    "SETOR_PAKSA", "SETOR_SITA", "SETOR_CEGAH", "SETOR_SPRINDRA",
    "JML_SURAT_TEGURAN", "JML_SURAT_PAKSA",
    "FLAG_PERNAH_DISITA", "FLAG_PERNAH_BLOKIR", "FLAG_RESPON_PENAGIHAN",
    "TGL_TEGURAN", "TGL_PENYAMPAIAN_SP", "TGL_BAPS",
]

NUM_FEATURES = [
    "TOTAL_TUNGGAKAN_POKOK", "LOG_TOTAL_TUNGGAKAN", "RATA_TUNGGAKAN",
    "MEDIAN_TUNGGAKAN", "MAX_TUNGGAKAN", "LOG_MAX_TUNGGAKAN",
    "JML_KETETAPAN", "JML_JENIS_PAJAK", "JML_JENIS_KETETAPAN",
    "UMUR_UTANG_RATA", "UMUR_UTANG_TERTUA", "SISA_DALUWARSA_TERDEKAT",
    "RATA_SELISIH_TAHUN_TERBIT",
    "RASIO_LAPOR_SPT_3THN", "FLAG_LAPOR_SPT_TERAKHIR", "FLAG_TANPA_DATA_SPT",
    "PEREDARAN_BRUTO", "LOG_PEREDARAN_BRUTO", "FLAG_PEREDARAN_NOL_KOSONG",
    "RASIO_TUNGGAKAN_PEREDARAN",
    "JML_CUSTOMER", "DPP_CUSTOMER", "LOG_DPP_CUSTOMER", "FAKTUR_CUSTOMER",
    "JML_SUPPLIER", "DPP_SUPPLIER", "LOG_DPP_SUPPLIER", "FAKTUR_SUPPLIER",
    "TOTAL_MITRA",
]
CAT_FEATURES = [
    "STS_WP", "JENIS_WP", "JENIS_KPP_BKM", "KD_KANWIL", "SEKTOR_KLU",
    "JENIS_PAJAK_DOMINAN", "KODE_KETETAPAN_DOMINAN",
]
FEATURES = NUM_FEATURES + CAT_FEATURES

assert not (set(FEATURES) & set(OUTCOME_COLS)), "Outcome leakage masuk FEATURES!"
assert "LABEL" not in FEATURES and "LABEL_PURITY" not in FEATURES

print(f"Fitur numerik: {len(NUM_FEATURES)} | kategorikal: {len(CAT_FEATURES)}")
print("Outcome dikecualikan:", OUTCOME_COLS)

## 8. 🔀 Split & Preprocessor

- 1 baris = 1 WP → split stratified biasa.
- Median-impute + scaling untuk numerik.
- Most-frequent + one-hot untuk kategorikal.
- Statistik preprocessing di-fit hanya pada train.

In [ ]:
X = wp[FEATURES].copy()
y = wp["LABEL"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CONFIG["test_size"], stratify=y, random_state=RNG,
)


def buat_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), NUM_FEATURES),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), CAT_FEATURES),
    ])

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print("Distribusi train:", (y_train.value_counts(normalize=True).sort_index()*100).round(1).to_dict())

## 9. ⚖️ Perbandingan Model (CV 5-fold)

`class_weight="balanced"` dipakai pada model yang mendukungnya karena kelas Sedang
lebih sedikit daripada Rendah/Tinggi. Dummy adalah baseline minimal.

In [ ]:
def buat_model(nama):
    if nama == "Dummy":
        return DummyClassifier(strategy="most_frequent")
    if nama == "LogReg":
        return LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RNG)
    if nama == "RandomForest":
        return RandomForestClassifier(n_estimators=400, class_weight="balanced", n_jobs=-1, random_state=RNG)
    if nama == "HistGB":
        return HistGradientBoostingClassifier(random_state=RNG)
    raise ValueError(nama)

models = ["Dummy", "LogReg", "RandomForest", "HistGB"]
cv = StratifiedKFold(CONFIG["cv_folds"], shuffle=True, random_state=RNG)
rows = []
for nama in models:
    pipe = Pipeline([("prep", buat_preprocessor()), ("clf", buat_model(nama))])
    sc = cross_validate(pipe, X_train, y_train, cv=cv,
                        scoring=["f1_macro", "balanced_accuracy", "accuracy"], n_jobs=1)
    rows.append({
        "model": nama,
        "f1_macro": sc["test_f1_macro"].mean(),
        "f1_std": sc["test_f1_macro"].std(),
        "balanced_accuracy": sc["test_balanced_accuracy"].mean(),
        "accuracy": sc["test_accuracy"].mean(),
    })
hasil_cv = pd.DataFrame(rows).set_index("model").sort_values("f1_macro", ascending=False)
display(hasil_cv.round(4))

fig, ax = plt.subplots(figsize=(6, 3))
hasil_cv["f1_macro"].sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Model kapasitas-only — CV F1-macro")
plt.tight_layout(); plt.show()

## 10. 🔧 Tuning Ringan Pemenang

Tuning kecil dilakukan hanya pada model terbaik CV, tetap mengoptimalkan `f1_macro`.

In [ ]:
param_grids = {
    "LogReg": {
        "clf__C": [0.01, 0.05, 0.1, 0.5, 1, 5, 10],
        "clf__class_weight": [None, "balanced"],
    },
    "RandomForest": {
        "clf__n_estimators": [300, 500, 800],
        "clf__max_depth": [None, 10, 20],
        "clf__min_samples_leaf": [1, 2, 5, 10],
        "clf__max_features": ["sqrt", 0.5],
    },
    "HistGB": {
        "clf__learning_rate": [0.03, 0.05, 0.1, 0.2],
        "clf__max_iter": [200, 400, 600],
        "clf__max_leaf_nodes": [15, 31, 63],
        "clf__min_samples_leaf": [10, 20, 40],
        "clf__l2_regularization": [0.0, 1.0],
    },
}

pemenang = hasil_cv.index[0]
if pemenang == "Dummy":
    pemenang = "LogReg"

search = RandomizedSearchCV(
    Pipeline([("prep", buat_preprocessor()), ("clf", buat_model(pemenang))]),
    param_distributions=param_grids[pemenang],
    n_iter=min(15, np.prod([len(v) for v in param_grids[pemenang].values()])),
    scoring="f1_macro",
    cv=StratifiedKFold(3, shuffle=True, random_state=RNG),
    random_state=RNG, n_jobs=1,
)
search.fit(X_train, y_train)
final_model = search.best_estimator_
print("Pemenang:", pemenang)
print(f"Best CV F1-macro: {search.best_score_:.4f}")
print("Parameter:", search.best_params_)

## 11. 🎯 Evaluasi Final

Nilai sekitar 0,5–0,6 jauh lebih realistis daripada V1 (~0,99), karena model tidak lagi
melihat jawaban (`SETOR_*`). Fokus interpretasi: apakah sinyal kapasitas benar-benar
membedakan ketertagihan.

In [ ]:
y_pred = final_model.predict(X_test)
print(classification_report(
    y_test, y_pred,
    target_names=[CONFIG["label_names"][i] for i in [0, 1, 2]],
))

print(f"F1-macro          : {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"Balanced accuracy : {balanced_accuracy_score(y_test, y_pred):.4f}")
print("Recall 0/1/2      :", recall_score(y_test, y_pred, labels=[0,1,2], average=None).round(3))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, normalize="true", cmap="Blues", ax=ax,
    display_labels=["Rendah", "Sedang", "Tinggi"],
)
ax.set_title("Confusion matrix V2 (kapasitas-only)")
plt.tight_layout(); plt.show()

## 12. 🔍 Analisis Kualitas Target & Segmen

- `LABEL_PURITY >= 0.75`: mayoritas kohir WP relatif konsisten.
- Purity rendah: satu WP sering membayar dengan mekanisme berbeda; target modus lebih noisy.
- Status WP dianalisis terpisah untuk memastikan NE/DE tidak tersembunyi oleh dominasi AKTIF.

In [ ]:
def metrik_segmen(mask, nama):
    yt = y_test[mask]
    yp = y_pred[mask]
    if len(yt) < 5 or yt.nunique() < 2:
        return None
    rec = recall_score(yt, yp, labels=[0,1,2], average=None, zero_division=0)
    return {"segmen": nama, "n": len(yt),
            "f1_macro": f1_score(yt, yp, average="macro", zero_division=0),
            "recall_Rendah": rec[0], "recall_Sedang": rec[1], "recall_Tinggi": rec[2]}

meta_test = wp.loc[X_test.index]
segmen = [
    metrik_segmen(np.ones(len(y_test), dtype=bool), "SEMUA"),
    metrik_segmen((meta_test["LABEL_PURITY"] >= 0.75).values, "Purity >= 0.75"),
    metrik_segmen((meta_test["LABEL_PURITY"] < 0.75).values, "Purity < 0.75"),
    metrik_segmen((meta_test["STS_WP"] == "AKTIF").values, "WP AKTIF"),
    metrik_segmen((meta_test["STS_WP"] != "AKTIF").values, "WP NON-AKTIF"),
]
display(pd.DataFrame([s for s in segmen if s]).round(3))

## 13. 🧠 Permutation Importance

Importance diukur pada test set: seberapa besar F1-macro turun saat satu fitur diacak.
Fitur outcome tidak mungkin muncul karena dilarang oleh kontrak §7.

In [ ]:
perm = permutation_importance(
    final_model, X_test, y_test, scoring="f1_macro",
    n_repeats=10, random_state=RNG, n_jobs=1,
)
importance = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 7))
importance.head(20)[::-1].plot(kind="barh", ax=ax, color="seagreen")
ax.set_title("Top-20 fitur kapasitas — permutation importance")
plt.tight_layout(); plt.show()
display(importance.head(15).round(4).to_frame("importance"))

## 14. 📋 Demo Ranking Triase

Skor probabilitas digunakan langsung di level WP:
- `P_Rendah` tinggi → indikasi sulit tertagih;
- `P_Tinggi` tinggi → quick win potensial.

Ranking berikut adalah demonstrasi pada test set, bukan daftar produksi.

In [ ]:
proba = final_model.predict_proba(X_test)
classes = final_model.named_steps["clf"].classes_
proba_df = pd.DataFrame(
    proba,
    columns=[f"P_{CONFIG['label_names'][c]}" for c in classes],
    index=X_test.index,
)
triase = wp.loc[X_test.index, ["NPWP16", "NAMA_WP", "STS_WP", "TOTAL_TUNGGAKAN_POKOK", "LABEL", "LABEL_PURITY"]].join(proba_df)

print("🔺 Indikasi ketertagihan rendah:")
display(triase.sort_values(["P_Rendah", "TOTAL_TUNGGAKAN_POKOK"], ascending=False).head(10).round(3))
print("✅ Quick win potensial:")
display(triase.sort_values(["P_Tinggi", "TOTAL_TUNGGAKAN_POKOK"], ascending=False).head(10).round(3))

## 15. 💾 Simpan Artefak V2

In [ ]:
os.makedirs(os.path.dirname(CONFIG["path_features"]), exist_ok=True)
os.makedirs(os.path.dirname(CONFIG["path_model"]), exist_ok=True)

wp.to_csv(CONFIG["path_features"], index=False)
joblib.dump(final_model, CONFIG["path_model"])

pred = triase.copy()
pred["PREDIKSI"] = y_pred
pred.to_csv(CONFIG["path_predictions"], index=False)

print(f"Features    : {CONFIG['path_features']} ({wp.shape[0]:,} WP × {wp.shape[1]} kolom)")
print(f"Model       : {CONFIG['path_model']}")
print(f"Predictions : {CONFIG['path_predictions']} ({len(pred):,} WP test)")

## 📌 Kesimpulan & Roadmap

**V2 saat ini:**
- memakai 195 ribu+ kohir untuk membentuk profil 2.873 WP;
- mempertahankan LABEL 0/1/2 sebagai outcome;
- kohir LABEL NULL memperkaya konteks, tidak menentukan target;
- model tidak melihat kolom pembentuk LABEL;
- performa V2 adalah baseline cross-sectional capacity-only, bukan klaim validasi prospektif.

**Pengembangan berikutnya:**
1. **Data aset (prioritas tertinggi):** rekening, kendaraan, properti, piutang/aset teridentifikasi, nilai aset — belum tersedia.
2. **Sengketa & reachability:** keberatan/banding aktif, alamat valid/tidak ditemukan, pailit/bubar/meninggal.
3. **Review target WP:** modus hitungan vs tertimbang nilai berbeda pada sekitar seperempat WP; keputusan domain diperlukan.
4. **Temporal validation:** bila tanggal pembayaran tersedia, bentuk fitur as-of-cutoff dan label outcome setelah cutoff.
5. **Model ordinal/biner:** bandingkan 3 kelas dengan `0 vs {1,2}` dan ordinal 0<1<2.
6. **Kalibrasi & threshold biaya:** prioritaskan recall Rendah atau expected collectible value sesuai kapasitas tim.